In [1]:
"""
Copyright (c) 2026 Baechul Kim
All rights reserved.

Author: Baechul Kim <baechul@gmail.com>
Date: April 15, 2026
Description: Latest raw sales data cleaning provcess 
License: MIT
"""

import pandas as pd
import numpy as np

# Load and Explore the Data
# I downloaded "Online Sales Data.csv" from public internet for Demo
data_path = 'data/Online Sales Data.csv'
df = pd.read_csv(data_path)

print('Raw data shape:', df.shape)        # (240, 9)
print('Columns', df.columns.tolist())     # ['Transaction ID', 'Date', 'Product Category', 'Product Name', 'Units Sold', 'Unit Price', 'Total Revenue', 'Region', 'Payment Method']
print('Missing Values', df.isna().sum())  # 0

Raw data shape: (240, 9)
Columns ['Transaction ID', 'Date', 'Product Category', 'Product Name', 'Units Sold', 'Unit Price', 'Total Revenue', 'Region', 'Payment Method']
Missing Values Transaction ID      0
Date                0
Product Category    0
Product Name        0
Units Sold          0
Unit Price          0
Total Revenue       0
Region              0
Payment Method      0
dtype: int64


In [ ]:
# Data Cleaning and Preprocessing
# This is what I usually do for data cleaning before training.

# 1) Standardize column names
original_columns = df.columns.tolist()
new_columns = [col.strip().lower().replace(' ', '_') for col in original_columns]
df.columns = new_columns

# 2) Remove duplicates
initial_rows = len(df)
df = df.drop_duplicates()
after_dedup_rows = len(df)
print(f'Rows before deduplication: {initial_rows}, after: {after_dedup_rows}')

# 3) Handle missing values
missing_summary = df.isna().sum()     # 0 luckily for the one I downloaded
print('Missing values after dedup:')
print(missing_summary)

# 4) Drop rows with invalid dates or missing revenue information
required_cols = ['date', 'product_category', 'total_revenue'] # baseline features for train/predict
df = df.dropna(subset=required_cols)

# 5) Convert to numeric columns if needed
numeric_cols = ['units_sold', 'unit_price', 'total_revenue']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 6) Then drop rows with invalid numeric values in baseline features
df = df.dropna(subset=numeric_cols)

# Final data summary after cleaning
print('Data shape after clean:', df.shape)

# I will be using this cleaned data as the latest sales data for prediction.
# In prod env, there will be a separate workflow to regulary precompute and 
# update it so agent's prediction is based on the latest sales data.  
df.to_csv('./data/sales.csv')

Rows before deduplication: 240, after: 240
Missing values after dedup:
transaction_id      0
date                0
product_category    0
product_name        0
units_sold          0
unit_price          0
total_revenue       0
region              0
payment_method      0
dtype: int64
Data shape after clean: (240, 9)


In [10]:
# Statistical Analysis (Not requested but just for fun :))
print('Descriptive statistics for numeric fields:')
print(df[['units_sold', 'unit_price', 'total_revenue']].describe())

print('\nCorrelation matrix:')
correlation = df[['units_sold', 'unit_price', 'total_revenue']].corr()
print(correlation)

print('\nAverage revenue per product sold by product category:')
avg_revenue_per_unit = (df.groupby('product_category')['total_revenue'].sum() / df.groupby('product_category')['units_sold'].sum()).sort_values(ascending=False)
print(avg_revenue_per_unit.head(3))

# Descriptive statistics for numeric fields:
#        units_sold   unit_price  total_revenue
# count  240.000000   240.000000     240.000000
# mean     2.158333   236.395583     335.699375
# std      1.322454   429.446695     485.804469
# min      1.000000     6.500000       6.500000
# 25%      1.000000    29.500000      62.965000
# 50%      2.000000    89.990000     179.970000
# 75%      3.000000   249.990000     399.225000
# max     10.000000  3899.990000    3899.990000

# (KIM's Note) See -0.3 between unit_price and units sold - 
# the more expense the less sold (that's what negative means) 
# Correlation matrix:
#                units_sold  unit_price  total_revenue
# units_sold       1.000000   -0.308583      -0.171151
# unit_price      -0.308583    1.000000       0.930350
# total_revenue   -0.171151    0.930350       1.000000

# Average revenue per product sold by product category:
# product_category
# Electronics        530.036515
# Home Appliances    316.036610
# Sports             162.801364
# dtype: float64

Descriptive statistics for numeric fields:
       units_sold   unit_price  total_revenue
count  240.000000   240.000000     240.000000
mean     2.158333   236.395583     335.699375
std      1.322454   429.446695     485.804469
min      1.000000     6.500000       6.500000
25%      1.000000    29.500000      62.965000
50%      2.000000    89.990000     179.970000
75%      3.000000   249.990000     399.225000
max     10.000000  3899.990000    3899.990000

Correlation matrix:
               units_sold  unit_price  total_revenue
units_sold       1.000000   -0.308583      -0.171151
unit_price      -0.308583    1.000000       0.930350
total_revenue   -0.171151    0.930350       1.000000

Average revenue per product sold by product category:
product_category
Electronics        530.036515
Home Appliances    316.036610
Sports             162.801364
dtype: float64
